# Pipeline: FCOS Tiled Inference

**Мета:** перевірити, чи покращує поділ зображення на фрагменти детекцію об'єктів без повторного навчання моделей. Нові tiled-прогнози FCOS об'єднуються з його повнокадровими прогнозами, після чого результат використовується в ансамблі **RetinaNet TTA + RT-DETR-L + FCOS** через Weighted Boxes Fusion (WBF).

**Початкова контрольна конфігурація:** ваги моделей `1/3, 1/3, 1/3`, WBF `IoU=0.65`, `conf_type="max"`, `skip_box_thr=0.02`. Її результат на повному validation set — **mAP = 0.186048** (для малих об'єктів `AP_small = 0.097091`).

## Підготовка середовища та вхідні дані

В оригінальних запусках використано **Kaggle з GPU**. Для відтворення експерименту потрібні такі Kaggle Inputs:

**Зауваження щодо повторного запуску та налаштування процесів:** у збереженій першій клітинці налаштувань `RUN_PILOT=True`, `RUN_FULL=True`, `RUN_TEST=False`, відповідно до мети можна міняти ці значення.

In [ ]:
from pathlib import Path
import os, sys, json, gzip, io, zipfile, math, hashlib, contextlib, time, random, shutil, subprocess, importlib.util
from collections import defaultdict
from dataclasses import dataclass
from itertools import product
from datetime import datetime, timezone

for module, package in [('pycocotools','pycocotools'), ('ensemble_boxes','ensemble-boxes')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ensemble_boxes import weighted_boxes_fusion
from tqdm.auto import tqdm
from IPython.display import display

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUT = WORK / 'ensemble_postprocessing_v2'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Exact paths can be set to disambiguate duplicate inputs.
DATA_ROOT_OVERRIDE = None
PREP_ROOT_OVERRIDE = None
SOURCE_OVERRIDES = {k: None for k in (
    'val_fcos','val_retina_tta','val_rtdetr',
    'test_fcos','test_retina_tta','test_rtdetr','fcos_weights')}

# GPU stages
RUN_PILOT = True
RUN_FULL = True
RUN_TEST = False

# Pilot uses ~241 frames from all validation sequences, evenly spaced.
PILOT_EVERY_N = 12
TILE_CONFIGS = [
    {'size': 640, 'overlap': 0.20},
]
FULL_TILE_CONFIG_INDEX = 0
TILE_SCORE_FACTORS = [0.70, 0.85, 1.00]
TILE_NMS_IOU = 0.50
TILE_EDGE_MARGIN = 0
FCOS_MIN_SIZE = 800
FCOS_MAX_SIZE = 1333
FCOS_NMS_THRESH = 0.60
PRED_SCORE_MIN = 0.02
PER_MODEL_MAX_DET = 300
OUTPUT_MAX_DET = 500
COCO_MAX_DET = 100
SAVE_EVERY_N_IMAGES = 20

LABELS = {1:'person',2:'bicycle',3:'car',4:'van',5:'truck',6:'tricycle',7:'bus',8:'motor'}
MODELS = ('retina_tta','rtdetr','fcos')
OUT = WORK / 'fcos_tiled_next_experiment'
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'cache').mkdir(exist_ok=True)
print('Output folder:', OUT)
print('GPU:', __import__('torch').cuda.is_available())
print('Pilot:', RUN_PILOT, '| Full:', RUN_FULL, '| Test:', RUN_TEST)
if (RUN_PILOT or RUN_FULL) and not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU in Notebook Settings before tiled inference.')

Output folder: /kaggle/working/fcos_tiled_next_experiment
GPU: True
Pilot: True | Full: True | Test: False


## 1. Пошук готових прогнозів моделей і ваг FCOS

Знаходимо вихідні prediction-файли для RetinaNet TTA, RT-DETR і FCOS, а також checkpoint FCOS, який використовується для inference на фрагментах зображень.

In [ ]:
@dataclass(frozen=True)
class Asset:
    path: Path
    member: str | None = None
    def label(self): return str(self.path) + (f'::{self.member}' if self.member else '')

ASSET_NAMES = {
    'val_fcos': ('val_fcos.json','val_fcos.json.gz'),
    'val_retina_tta': ('val_retina_tta.json','val_retina_tta.json.gz'),
    'val_rtdetr': ('val_rtdetr.json','val_rtdetr.json.gz'),
    'test_fcos': ('test_fcos.json','test_fcos.json.gz'),
    'test_retina_tta': ('test_retina_tta.json','test_retina_tta.json.gz'),
    'test_rtdetr': ('test_rtdetr.json','test_rtdetr.json.gz'),
    'fcos_weights': ('model_weights.pth',),
}

def scan_input_files():
    wanted={n for names in ASSET_NAMES.values() for n in names}
    matches={k:[] for k in ASSET_NAMES}
    archives=[]
    for p in INPUT.rglob('*'):
        if not p.is_file(): continue
        if p.suffix.lower()=='.zip': archives.append(p)
        if p.name in wanted:
            for k,v in ASSET_NAMES.items():
                if p.name in v: matches[k].append(Asset(p))
    for zp in archives:
        try:
            with zipfile.ZipFile(zp) as z:
                for name in z.namelist():
                    short=Path(name).name
                    for k,v in ASSET_NAMES.items():
                        if short in v: matches[k].append(Asset(zp,name))
        except (OSError, zipfile.BadZipFile) as e:
            print('Cannot scan:',zp,e)
    return matches, archives

FOUND, ARCHIVES = scan_input_files()
for key, items in FOUND.items():
    print(f'[{key}] {len(items)} candidate(s)')
    for a in items[:6]: print('  ',a.label())
print('ZIP archives:',len(ARCHIVES))

def choose_asset(key, required=False):
    override=SOURCE_OVERRIDES.get(key)
    if override:
        s=str(override)
        if '::' in s:
            p,member=s.split('::',1); a=Asset(Path(p),member)
        else:
            a=Asset(Path(s))
        if not a.path.is_file(): raise FileNotFoundError(a.label())
        return a
    choices=FOUND[key]
    physical=[a for a in choices if a.member is None]
    if len(physical)==1:
        choices=physical
    elif len(physical)>1:
        choices=physical
    if len(choices)>1:
        raise ValueError(f'Ambiguous {key}. Set SOURCE_OVERRIDES[{key!r}] to one exact path or zip::member.\n' + '\n'.join(a.label() for a in choices))
    if required and not choices:
        raise FileNotFoundError(f'Missing {key}. Add prior final-ensemble outputs containing {ASSET_NAMES[key]}')
    return choices[0] if choices else None

def read_asset_bytes(asset):
    if asset.member:
        with zipfile.ZipFile(asset.path) as z:
            return z.read(asset.member)
    return asset.path.read_bytes()

def read_asset_json(asset):
    b=read_asset_bytes(asset)
    if str(asset.member or asset.path.name).endswith('.gz'):
        b=gzip.decompress(b)
    return json.loads(b)

def materialize_weight(asset):
    if asset.member is None: return asset.path
    dest=WORK/'fcos_weights_from_input.pth'
    with zipfile.ZipFile(asset.path) as z, z.open(asset.member) as src, open(dest,'wb') as dst:
        shutil.copyfileobj(src,dst)
    return dest

for k in ('val_fcos','val_retina_tta','val_rtdetr'):
    choose_asset(k,required=True)
print('All three full-validation prediction files found.')

[val_fcos] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/val_fcos.json
[val_retina_tta] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/val_retina_tta.json
[val_rtdetr] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/val_rtdetr.json
[test_fcos] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/test_fcos.json
[test_retina_tta] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/test_retina_tta.json
[test_rtdetr] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/test_rtdetr.json
[fcos_weights] 1 candidate(s)
   /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/model_weights.pth
ZIP archives: 0
All three full-validation prediction files found.


## 2. Підготовка ground truth та COCO evaluation

Використовуємо той самий принцип оцінювання, що й для попереднього WBF: **2846 validation-кадрів**, класи `1..8`, врахування ignored regions як `crowd` та `COCO maxDets=100`. Завдяки однаковій метриці результати можна порівнювати.

In [3]:
def discover_data_root():
    if DATA_ROOT_OVERRIDE: return Path(DATA_ROOT_OVERRIDE)
    candidates=sorted({p.parent.parent for p in INPUT.rglob('annotations_coco.json')
        if p.parent.name=='val' and (p.parent.parent/'train/annotations_coco.json').is_file()})
    if len(candidates)!=1: raise ValueError(f'Expected one DATA_ROOT, found {candidates}. Set DATA_ROOT_OVERRIDE.')
    return candidates[0]

def discover_prep_root():
    if PREP_ROOT_OVERRIDE: return Path(PREP_ROOT_OVERRIDE)
    candidates=sorted({p.parent for p in INPUT.rglob('val.json') if (p.parent/'train.json').is_file()})
    if len(candidates)!=1: raise ValueError(f'Expected one PREP_ROOT, found {candidates}. Set PREP_ROOT_OVERRIDE.')
    return candidates[0]

DATA_ROOT=discover_data_root()
PREP_ROOT=discover_prep_root()
orig=json.loads((DATA_ROOT/'val/annotations_coco.json').read_text())
original={int(im['id']):im for im in orig['images']}
prep=json.loads((PREP_ROOT/'val.json').read_text())
assert len(prep)==len(original), f'Validation sizes differ {len(prep)} vs {len(original)}'
val_records=[]
for row in prep:
    iid=int(row['image_id']); o=original[iid]
    labels=[int(x) for x in row['labels']]
    assert len(row['boxes'])==len(labels) and set(labels).issubset(LABELS)
    val_records.append({'image_id':iid,'file_name':row.get('file_name',o['file_name']),
        'width':int(o['width']),'height':int(o['height']),
        'boxes':row['boxes'],'labels':labels,'ignored_boxes':row.get('ignored_boxes',[])})
val_records.sort(key=lambda r:(str(r['file_name']).rsplit('/',1)[0],str(r['file_name'])))
VAL_IDS={r['image_id'] for r in val_records}
assert len(VAL_IDS)==len(val_records)
print('DATA_ROOT:',DATA_ROOT)
print('PREP_ROOT:',PREP_ROOT)
print('Official validation frames:',len(val_records))

def find_image(split,filename):
    rel=Path(str(filename).replace('\\','/').lstrip('/'))
    if '..' in rel.parts: raise ValueError(f'Unsafe path {filename!r}')
    opts=[DATA_ROOT/split/rel,DATA_ROOT/split/'images'/rel,DATA_ROOT/rel]
    if rel.parts and rel.parts[0]==split:
        opts.extend([DATA_ROOT/rel, DATA_ROOT/split/Path(*rel.parts[1:])])
    for p in opts:
        if p.is_file():return p
    raise FileNotFoundError(f'{split}: {filename}')

def xyxy_to_xywh(b):
    x1,y1,x2,y2=map(float,b)
    return [x1,y1,max(0.0,x2-x1),max(0.0,y2-y1)]

def xywh_to_xyxy(b):
    x,y,w,h=map(float,b)
    return [x,y,x+w,y+h]

def build_gt(rows):
    images=[]; annotations=[]
    for r in rows:
        iid=int(r['image_id'])
        images.append({'id':iid,'file_name':str(r['file_name']),
            'width':int(r['width']),'height':int(r['height'])})
        for b,cid in zip(r['boxes'],r['labels']):
            bb=xyxy_to_xywh(b)
            if bb[2]>0 and bb[3]>0:
                annotations.append({'id':len(annotations)+1,'image_id':iid,
                    'category_id':int(cid),'bbox':bb,'area':bb[2]*bb[3],'iscrowd':0})
        for b in r['ignored_boxes']:
            bb=xyxy_to_xywh(b)
            if bb[2]<=0 or bb[3]<=0:continue
            for cls in LABELS:
                annotations.append({'id':len(annotations)+1,'image_id':iid,
                    'category_id':cls,'bbox':bb,'area':bb[2]*bb[3],
                    'iscrowd':1,'ignore':1})
    gt=COCO();gt.dataset={'images':images,'annotations':annotations,
        'categories':[{'id':k,'name':v} for k,v in LABELS.items()], 'info':{},'licenses':[]}
    with contextlib.redirect_stdout(io.StringIO()):gt.createIndex()
    return gt

COCO_FULL=build_gt(val_records)

def coco_metrics(preds,gt):
    allowed=set(gt.getImgIds())
    for p in preds:
        assert type(p['image_id']) is int and p['image_id'] in allowed, ('bad COCO image ID', p['image_id'])
        assert int(p['category_id']) in LABELS
    with contextlib.redirect_stdout(io.StringIO()):
        if preds: dt=gt.loadRes(preds)
        else:
            dt=COCO();dt.dataset={'images':gt.dataset['images'], 'annotations':[],
                'categories':gt.dataset['categories'],'info':{},'licenses':[]};dt.createIndex()
        ev=COCOeval(gt,dt,'bbox')
        ev.params.imgIds=sorted(allowed)
        ev.params.catIds=list(LABELS)
        ev.params.maxDets=[1,10,COCO_MAX_DET]
        ev.evaluate();ev.accumulate();ev.summarize()
    return dict(zip(('mAP','AP50','AP75','AP_small','AP_medium','AP_large'),map(float,ev.stats[:6])))

def normalize_preds(items,split='val'):
    if isinstance(items,dict): items=items.get('predictions')
    if not isinstance(items,list):raise TypeError('Expected list of COCO prediction dictionaries')
    output=[]
    for p in items:
        iid=int(p['image_id']) if split=='val' else str(p['image_id'])
        cls=int(p['category_id'])
        if cls not in LABELS: raise ValueError('Expected COCO class 1..8; got '+str(cls))
        b=list(map(float,p['bbox'])); score=float(p['score'])
        if len(b)!=4 or not all(map(math.isfinite,b+[score])) or not 0<=score<=1:
            raise ValueError('Invalid bbox or confidence')
        if b[2]>0 and b[3]>0 and score>=PRED_SCORE_MIN:
            output.append({'image_id':iid,'category_id':cls,'bbox':b,'score':score})
    if split=='val' and not {p['image_id'] for p in output}.issubset(VAL_IDS):
        raise ValueError('Validation predictions contain foreign image IDs')
    return output

def grouped(preds):
    d=defaultdict(list)
    for p in preds:d[str(p['image_id'])].append(p)
    return d

def write_gz(path,rows):
    with gzip.open(path,'wt',encoding='utf8',compresslevel=5) as f:
        json.dump(rows,f,separators=(',',':'))

GT_REFERENCE_MMAP=0.179263
print('COCO evaluator ready; consistent with final-ensemble.ipynb.')

DATA_ROOT: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
PREP_ROOT: /kaggle/input/datasets/olenadovbenchuk/preprocessing-results
Official validation frames: 2846
COCO evaluator ready; consistent with final-ensemble.ipynb.


## 3. Відтворення результатів моделей та початкового WBF

Завантажуємо збережені predictions і порівнюємо кожен детектор із початковим ансамблем. Це перевірка того, що експеримент спирається на ті самі вихідні прогнози.

In [5]:
RAW_VAL={}
SOURCE_SIGNATURES={}
for model in MODELS:
    asset=choose_asset('val_'+model,required=True)
    raw=read_asset_bytes(asset)
    SOURCE_SIGNATURES['val_'+model]=hashlib.sha256(raw).hexdigest()
    payload = read_asset_json(asset)
    RAW_VAL[model]=normalize_preds(payload,'val')
    print(model, len(RAW_VAL[model]),'predictions | source:',asset.label())

GROUPS={m:grouped(RAW_VAL[m]) for m in MODELS}
base_rows=[]
for m in MODELS:
    met=coco_metrics(RAW_VAL[m],COCO_FULL)
    base_rows.append({'candidate':m,**met})
base_df=pd.DataFrame(base_rows).sort_values('mAP',ascending=False)
base_df.to_csv(OUT/'individual_models.csv',index=False)
display(base_df)

# Original ensemble is included in every search, not just a historical printed number.
BASE_SPEC={'name':'baseline_equal_wbf_055_avg_002',
    'models':list(MODELS),'weights':[1/3,1/3,1/3],
    'iou':0.55,'conf_type':'avg','skip_thr':0.02}
print('Raw source signatures saved (SHA256 of the compressed input files).')

retina_tta 571363 predictions | source: /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/val_retina_tta.json
rtdetr 849015 predictions | source: /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/val_rtdetr.json
fcos 203039 predictions | source: /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/val_fcos.json


,candidate,mAP,AP50,AP75,AP_small,AP_medium,AP_large
0,retina_tta,0.161712,0.317458,0.145288,0.068371,0.152064,0.306306
1,rtdetr,0.144829,0.285058,0.129428,0.092455,0.143043,0.238665
2,fcos,0.132367,0.271539,0.114144,0.065504,0.119132,0.277170


Raw source signatures saved (SHA256 of the compressed input files).


In [6]:
def wbf_one_image(rec,spec,groups):
    iid=rec['image_id']; key=str(iid)
    width=float(rec['width']);height=float(rec['height'])
    boxes_list=[];scores_list=[];labels_list=[]
    for model in spec['models']:
        parts=sorted(groups[model].get(key,[]),key=lambda z:z['score'],reverse=True)[:PER_MODEL_MAX_DET]
        bb=[];ss=[];ll=[]
        for p in parts:
            x1,y1,x2,y2=xywh_to_xyxy(p['bbox'])
            x1=max(0.,min(width,x1));y1=max(0.,min(height,y1))
            x2=max(0.,min(width,x2));y2=max(0.,min(height,y2))
            if x2<=x1 or y2<=y1:continue
            bb.append([x1/width,y1/height,x2/width,y2/height])
            ss.append(float(p['score']));ll.append(int(p['category_id']))
        boxes_list.append(bb);scores_list.append(ss);labels_list.append(ll)
    if not any(scores_list):return []
    fb,fs,fl=weighted_boxes_fusion(
        boxes_list,scores_list,labels_list,
        weights=spec['weights'],iou_thr=float(spec['iou']),
        skip_box_thr=float(spec['skip_thr']),
        conf_type=spec['conf_type'])
    output=[]
    for i in np.argsort(-np.asarray(fs))[:OUTPUT_MAX_DET]:
        x1,y1,x2,y2=fb[i]
        box=[x1*width,y1*height,(x2-x1)*width,(y2-y1)*height]
        conf=float(fs[i]);cls=int(fl[i])
        if cls in LABELS and box[2]>0 and box[3]>0 and conf>=PRED_SCORE_MIN:
            output.append({'image_id':iid if isinstance(iid,str) else int(iid),
                'category_id':cls,'bbox':list(map(float,box)),'score':conf})
    return output

def wbf_records(records,spec,groups=GROUPS,progress=False):
    iterator=tqdm(records,desc=spec['name'],leave=False) if progress else records
    return [p for r in iterator for p in wbf_one_image(r,spec,groups)]

# Unit check: coordinates, image IDs and original baseline value
probe=wbf_one_image(val_records[0],BASE_SPEC,GROUPS)
assert all(type(p['image_id']) is int and p['category_id'] in LABELS for p in probe)
base_fused=wbf_records(val_records,BASE_SPEC)
base_fused_metrics=coco_metrics(base_fused,COCO_FULL)
print('Recomputed ORIGINAL WBF baseline:',base_fused_metrics)
print('Historical reference:',GT_REFERENCE_MMAP)
if abs(base_fused_metrics['mAP']-GT_REFERENCE_MMAP)>0.002:
    raise RuntimeError('Baseline not reproduced. STOP: wrong raw predictions or metric protocol. Verify checkpoint/version, labels and val.json.')
print('Baseline reproduced. Safe to tune post-processing.')

Recomputed ORIGINAL WBF baseline: {'mAP': 0.17926342230669653, 'AP50': 0.36320397245047514, 'AP75': 0.15675078981687562, 'AP_small': 0.10117647413352358, 'AP_medium': 0.17082082983353042, 'AP_large': 0.33072379586867584}
Historical reference: 0.179263
Baseline reproduced. Safe to tune post-processing.


In [7]:
# Best WBF discovered in the uploaded 640/768-independent postprocessing notebook.
BEST_SPEC = {
    'name':'wbf_r33_t33_f33_i65_max_s02',
    'models':list(MODELS),'weights':[1/3,1/3,1/3],
    'iou':0.65,'conf_type':'max','skip_thr':0.02,
}
EXPECTED_MMAP = 0.18604802512506466
NEW_BASELINE = wbf_records(val_records,BEST_SPEC,GROUPS)
NEW_BASELINE_METRICS = coco_metrics(NEW_BASELINE,COCO_FULL)
print('Current WBF winner:',NEW_BASELINE_METRICS)
assert abs(NEW_BASELINE_METRICS['mAP']-EXPECTED_MMAP)<0.002, (
    'Cannot reproduce the current WBF winner; check attached predictions and COCO protocol!')
print('Ready for tiled experiment.')

Current WBF winner: {'mAP': 0.18604802512506466, 'AP50': 0.3632484219180758, 'AP75': 0.16873251524138222, 'AP_small': 0.09709072405653435, 'AP_medium': 0.1773813504055623, 'AP_large': 0.3272595400921645}
Ready for tiled experiment.


**Проміжний висновок, щодо результататів до застосування tiling**

Початкові детектори мають різну якість на validation: `RetinaNet TTA = 0.161712`, `RT-DETR = 0.144829`, `FCOS = 0.132367` mAP. Звичайний WBF дав `0.179263`, а вже налаштована раніше конфігурація WBF — **`0.186048`**.

## 4. FCOS Tiled Inference

Зображення розбивається на фрагменти **640×640 пікселів** із **20% перекриттям**. FCOS окремо обробляє кожен фрагмент, а отримані bounding boxes переводяться в координати початкового кадру.

Водночас **готові повнокадрові прогнози FCOS повторно не обчислюються**. Нові tiled-прогнози об'єднуються з ними через class-aware NMS. Щоб контролювати вплив додаткових детекцій, до їхніх `confidence scores` застосовуємо `factor ∈ {0.70, 0.85, 1.00}`.


In [8]:
def tiled_origins(length,tile,overlap):
    if length<=tile: return [0]
    step=max(1,int(round(tile*(1-overlap))))
    starts=list(range(0,length-tile+1,step))
    end=length-tile
    if starts[-1]!=end: starts.append(end)
    return sorted(set(starts))

def tile_windows(width,height,tile,overlap):
    return [(x,y,min(x+tile,width),min(y+tile,height))
            for y in tiled_origins(height,tile,overlap)
            for x in tiled_origins(width,tile,overlap)]

assert tile_windows(640,640,640,0.2)==[(0,0,640,640)]
assert max(x2 for x1,y1,x2,y2 in tile_windows(1920,1080,640,0.2))==1920
assert max(y2 for x1,y1,x2,y2 in tile_windows(1920,1080,640,0.2))==1080

import torch
import torchvision
from PIL import Image
from torchvision.ops import batched_nms
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSClassificationHead

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''): h.update(chunk)
    return h.hexdigest()

def checkpoint_source():
    src=choose_asset('fcos_weights',required=True)
    return src, materialize_weight(src)

def create_fcos(checkpoint):
    model=fcos_resnet50_fpn(weights=None,weights_backbone=None,num_classes=9,
        min_size=FCOS_MIN_SIZE,max_size=FCOS_MAX_SIZE,
        score_thresh=PRED_SCORE_MIN,nms_thresh=FCOS_NMS_THRESH,
        detections_per_img=PER_MODEL_MAX_DET)
    nanchors=model.anchor_generator.num_anchors_per_location()[0]
    model.head.classification_head=FCOSClassificationHead(
        model.backbone.out_channels,nanchors,9)
    payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
    state=payload['model'] if isinstance(payload,dict) and 'model' in payload else payload
    model.load_state_dict(state,strict=True)
    model=model.to(DEVICE).eval()
    print('FCOS loaded strictly:',checkpoint,'on',DEVICE)
    return model

def single_image_tiles(rec,model,split,cfg):
    path=find_image(split,rec['file_name'])
    with Image.open(path) as im: frame=np.asarray(im.convert('RGB'))
    h,w=frame.shape[:2]
    assert (w,h)==(int(rec['width']),int(rec['height'])), ('Image size mismatch',path)
    if w<=cfg['size'] and h<=cfg['size']: return []  # base FCOS already covers this image
    result=[]
    for x1,y1,x2,y2 in tile_windows(w,h,cfg['size'],cfg['overlap']):
        # Skip windows identical to the original full-frame image.
        if (x1,y1,x2,y2)==(0,0,w,h): continue
        arr=np.ascontiguousarray(frame[y1:y2,x1:x2,:])
        tensor=torch.from_numpy(arr).permute(2,0,1).float().div_(255.0).to(DEVICE)
        with torch.inference_mode():
            pred=model([tensor])[0]
        boxes=pred['boxes'].detach().cpu().numpy()
        scores=pred['scores'].detach().cpu().numpy()
        labels=pred['labels'].detach().cpu().numpy()
        for b,s,cls in zip(boxes,scores,labels):
            cls=int(cls);s=float(s)
            if cls not in LABELS or s<PRED_SCORE_MIN:continue
            a,c,d,e=map(float,b)
            if TILE_EDGE_MARGIN:
                # Suppress partial boxes near INTERIOR tile edges only.
                if (x1>0 and a<TILE_EDGE_MARGIN) or (y1>0 and c<TILE_EDGE_MARGIN) or (x2<w and (x2-x1-d)<TILE_EDGE_MARGIN) or (y2<h and (y2-y1-e)<TILE_EDGE_MARGIN):
                    continue
            a=max(0.,min(float(w),a+x1));d=max(0.,min(float(w),d+x1))
            c=max(0.,min(float(h),c+y1));e=max(0.,min(float(h),e+y1))
            if d>a and e>c:
                result.append({'image_id':int(rec['image_id']) if split=='val' else str(rec['image_id']),
                    'category_id':cls,'bbox':[a,c,d-a,e-c],'score':s})
    return result

print('GPU:',torch.cuda.is_available(),'| Example tiles on 1920×1080:',
      len(tile_windows(1920,1080,640,0.2)))

GPU: True | Example tiles on 1920×1080: 8


## 5. Кешування результатів inference

Для кожної конфігурації tiles існує окремий кеш. Проміжні результати зберігаються кожні **20 кадрів**.

In [9]:
def cache_paths(split,cfg):
    stem=f'{split}_tiles_s{cfg["size"]}_o{round(cfg["overlap"]*100)}_e{TILE_EDGE_MARGIN}'
    return OUT/'cache'/f'{stem}.json.gz',OUT/'cache'/f'{stem}.contract.json'

def cache_contract(split,cfg):
    source,checkpoint=checkpoint_source()
    raw_asset=choose_asset(f'{split}_fcos',required=True)
    contract={
        'split':split,'tile_size':cfg['size'],'overlap':cfg['overlap'],
        'tile_edge_margin':TILE_EDGE_MARGIN,'model_min_size':FCOS_MIN_SIZE,
        'model_max_size':FCOS_MAX_SIZE,'score_min':PRED_SCORE_MIN,
        'nms_threshold':FCOS_NMS_THRESH,'max_det':PER_MODEL_MAX_DET,
        'source_checkpoint_sha256':sha256_file(checkpoint),
        'full_frame_fcos_sha256':hashlib.sha256(read_asset_bytes(raw_asset)).hexdigest(),
        'torchvision_version':torchvision.__version__,
    }
    signature=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    return contract,signature,checkpoint

def write_cache_atomic(path,payload):
    tmp=path.with_suffix('.tmp.gz')
    with gzip.open(tmp,'wt',encoding='utf-8',compresslevel=4) as fp:
        json.dump(payload,fp,separators=(',',':'))
    os.replace(tmp,path)

def tiles_for_records(records,split,cfg):
    if not torch.cuda.is_available():
        raise RuntimeError('Please enable a Kaggle GPU (T4/P100/L4) for tiled inference.')
    path,guard=cache_paths(split,cfg)
    contract,sig,ckpt=cache_contract(split,cfg)
    if path.exists() and not guard.exists():
        raise RuntimeError('Tile cache exists without a contract: check/remove it manually.')
    if guard.exists() and json.loads(guard.read_text()).get('signature')!=sig:
        raise RuntimeError('Cached tiles belong to another checkpoint/config. Change OUT or remove the stale cache.')
    stored={}
    if path.exists():
        with gzip.open(path,'rt',encoding='utf8') as f:stored=json.load(f)
    missing=[r for r in records if str(r['image_id']) not in stored]
    print(f'{split} {cfg}: {len(stored)} cached, {len(missing)} missing of {len(records)} requested')
    if missing:
        model=create_fcos(ckpt)
        try:
            for n,rec in enumerate(tqdm(missing,desc=f'Tiling {split} {cfg}'),1):
                stored[str(rec['image_id'])]=single_image_tiles(rec,model,split,cfg)
                if n%SAVE_EVERY_N_IMAGES==0 or n==len(missing):
                    guard.write_text(json.dumps({'signature':sig,'contract':contract},indent=2))
                    write_cache_atomic(path,stored)
        finally:
            del model
            torch.cuda.empty_cache()
    elif not path.exists():
        write_cache_atomic(path,stored)
        guard.write_text(json.dumps({'signature':sig,'contract':contract},indent=2))
    wanted={str(r['image_id']) for r in records}
    return [p for k in sorted(wanted) for p in stored.get(k,[])]

def merge_full_plus_tiles(records,full_fcos_group,tile_preds,scale=1.0,nms_iou=TILE_NMS_IOU):
    tiles=grouped(tile_preds)
    merged=[]
    for rec in records:
        iid=rec['image_id'];key=str(iid); W=float(rec['width']);H=float(rec['height'])
        origins=[(p,False) for p in full_fcos_group.get(key,[])]+[(p,True) for p in tiles.get(key,[])]
        boxes=[];scores=[];classes=[]
        for p,is_tile in origins:
            s=float(p['score'])*(float(scale) if is_tile else 1.0)
            if s<PRED_SCORE_MIN:continue
            x1,y1,x2,y2=xywh_to_xyxy(p['bbox'])
            x1=max(0.,min(W,x1));x2=max(0.,min(W,x2))
            y1=max(0.,min(H,y1));y2=max(0.,min(H,y2))
            if x2<=x1 or y2<=y1:continue
            boxes.append([x1,y1,x2,y2]);scores.append(min(1.,s));classes.append(int(p['category_id']))
        if not boxes:continue
        keep=batched_nms(torch.tensor(boxes,dtype=torch.float32),
            torch.tensor(scores,dtype=torch.float32),torch.tensor(classes,dtype=torch.int64),nms_iou)
        for k in keep[:PER_MODEL_MAX_DET].tolist():
            merged.append({'image_id':iid if isinstance(iid,str) else int(iid),
                'category_id':classes[k],'bbox':xyxy_to_xywh(boxes[k]),'score':scores[k]})
    return merged

# Non-GPU smoke check for merge typing and geometry.
mock=[{'image_id':1,'width':100,'height':100}]
empty=merge_full_plus_tiles(mock,{},[])
assert empty==[]
print('Incremental cache + full-frame/tiles fusion ready.')

Incremental cache + full-frame/tiles fusion ready.


## 6. Pilot Validation (241 кадр)

Спочатку перевіряємо tiled inference на вибраних кадрах із validation-послідовностей. Порівнюємо два рівні:

- **FCOS:** звичайні повнокадрові predictions проти `FCOS full + tiles`.
- **Ансамбль:** звичайний WBF проти `WBF + tiled FCOS`.

У кожному випадку перевіряємо три `factor` для нових tiled-прогнозів: `0.70`, `0.85`, `1.00`. Pilot використовується для попереднього відбору, але остаточний висновок робимо за full validation.

In [10]:
by_seq=defaultdict(list)
for r in val_records: by_seq[str(r['file_name']).rsplit('/',1)[0]].append(r)
PILOT_ROWS=sorted([r for _,group in sorted(by_seq.items()) for r in group[::PILOT_EVERY_N]],key=lambda r:r['image_id'])
PILOT_IDS={int(r['image_id']) for r in PILOT_ROWS}
COCO_PILOT=build_gt(PILOT_ROWS)
print('Pilot frames:',len(PILOT_ROWS),'of',len(val_records))

PILOT_METRICS_FILE=OUT/'tiled_pilot_comparison.csv'
PILOT_RESULTS=[]
if RUN_PILOT:
    baseline_on_pilot=wbf_records(PILOT_ROWS,BEST_SPEC,GROUPS)
    PILOT_RESULTS.append({'name':'Triple original WBF','size':0,'overlap':0,'factor':0,
        **coco_metrics(baseline_on_pilot,COCO_PILOT)})
    base_fcos_pilot=[p for p in RAW_VAL['fcos'] if p['image_id'] in PILOT_IDS]
    PILOT_RESULTS.append({'name':'FCOS original','size':0,'overlap':0,'factor':0,
        **coco_metrics(base_fcos_pilot,COCO_PILOT)})
    for cfg in TILE_CONFIGS:
        crops=tiles_for_records(PILOT_ROWS,'val',cfg)
        for factor in TILE_SCORE_FACTORS:
            modified=merge_full_plus_tiles(PILOT_ROWS,GROUPS['fcos'],crops,scale=factor)
            mgrp=dict(GROUPS);mgrp['fcos']=grouped(modified)
            fusion=wbf_records(PILOT_ROWS,BEST_SPEC,mgrp)
            PILOT_RESULTS.append({'name':'FCOS full+tiles','size':cfg['size'],
                'overlap':cfg['overlap'],'factor':factor,
                **coco_metrics(modified,COCO_PILOT)})
            PILOT_RESULTS.append({'name':'Triple WBF with tiled FCOS','size':cfg['size'],
                'overlap':cfg['overlap'],'factor':factor,
                **coco_metrics(fusion,COCO_PILOT)})
    pilot_df=pd.DataFrame(PILOT_RESULTS)
    pilot_df.to_csv(PILOT_METRICS_FILE,index=False)
    display(pilot_df.sort_values('mAP',ascending=False))
    print('Saved:',PILOT_METRICS_FILE)
else:
    print('Pilot is disabled; existing output:',PILOT_METRICS_FILE.exists())
    if PILOT_METRICS_FILE.exists():display(pd.read_csv(PILOT_METRICS_FILE).sort_values('mAP',ascending=False))

Pilot frames: 241 of 2846
val {'size': 640, 'overlap': 0.2}: 0 cached, 241 missing of 241 requested
FCOS loaded strictly: /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/model_weights.pth on cuda


Tiling val {'size': 640, 'overlap': 0.2}:   0%|          | 0/241 [00:00<?, ?it/s]

,name,size,overlap,factor,mAP,AP50,AP75,AP_small,AP_medium,AP_large
3,Triple WBF with tiled FCOS,640,0.2,0.70,0.200599,0.390145,0.180147,0.114783,0.188559,0.345595
5,Triple WBF with tiled FCOS,640,0.2,0.85,0.198244,0.386154,0.178927,0.105919,0.187493,0.342390
0,Triple original WBF,0,0.0,0.00,0.197136,0.376207,0.179214,0.116503,0.183074,0.349857
7,Triple WBF with tiled FCOS,640,0.2,1.00,0.192407,0.374708,0.173348,0.099840,0.183968,0.336847
2,FCOS full+tiles,640,0.2,0.70,0.157036,0.329635,0.132963,0.078477,0.147010,0.291784
4,FCOS full+tiles,640,0.2,0.85,0.153619,0.323899,0.130784,0.079126,0.143711,0.289762
1,FCOS original,0,0.0,0.00,0.141460,0.282938,0.126166,0.067622,0.126580,0.285397
6,FCOS full+tiles,640,0.2,1.00,0.139489,0.303714,0.114616,0.079203,0.133786,0.277665


Saved: /kaggle/working/fcos_tiled_next_experiment/tiled_pilot_comparison.csv


### Результати Pilot Validation

| Конфігурація | mAP (241 кадр) | AP_small |
|---|---:|---:|
| FCOS original | 0.141460 | 0.067622 |
| FCOS full + tiles, factor 0.70 | **0.157036** | **0.078477** |
| WBF original | 0.197136 | **0.116503** |
| WBF + tiled FCOS, factor 0.70 | **0.200599** | 0.114783 |

`factor=0.70` підвищив pilot mAP як для самого FCOS, так і для ансамблю. `AP_small` ансамблю на pilot трохи **знизився**, хоча mAP загалом підвищився, тому не можна стверджувати, що на всіх метриках для малих об'єктів стало краще, потрібна перевірка на повній валідації.

## 7. Full Validation (2846 кадрів)

Після pilot запускаємо tiled FCOS на **повному офіційному validation set**. Частина кадрів уже є в кеші pilot, тому повторно обчислюються лише відсутні.

Для фінального порівняння залишаємо контрольний WBF та три варіанти `factor`. 

In [11]:
FULL_RESULTS_FILE=OUT/'tiled_full_comparison.csv'
SELECT_FILE=OUT/'tiled_final_selection.json'
FINAL_SELECTION=None
if RUN_FULL:
    assert PILOT_METRICS_FILE.exists(), 'Run pilot before full validation.'
    cfg=TILE_CONFIGS[FULL_TILE_CONFIG_INDEX]
    tiles_full=tiles_for_records(val_records,'val',cfg)
    FULL_RESULTS=[{'name':'Original WBF winner','size':0,'overlap':0,'factor':0,
                   **NEW_BASELINE_METRICS}]
    for factor in TILE_SCORE_FACTORS:
        modified=merge_full_plus_tiles(val_records,GROUPS['fcos'],tiles_full,scale=factor)
        groups=dict(GROUPS);groups['fcos']=grouped(modified)
        triple=wbf_records(val_records,BEST_SPEC,groups,progress=True)
        met=coco_metrics(triple,COCO_FULL)
        FULL_RESULTS.append({'name':'WBF + tiled FCOS','size':cfg['size'],
            'overlap':cfg['overlap'],'factor':factor,**met})
        write_gz(OUT/f'val_triple_tiled_s{cfg["size"]}_f{int(factor*100)}.json.gz',triple)
        print('FULL factor',factor,met)
    full_df=pd.DataFrame(FULL_RESULTS).sort_values(['mAP','AP_small'],ascending=False)
    full_df.to_csv(FULL_RESULTS_FILE,index=False)
    display(full_df)
    winner=full_df.iloc[0].to_dict()
    FINAL_SELECTION={'selected':winner,'original_wbf':BEST_SPEC,
        'baseline_mAP':EXPECTED_MMAP,'n_val':len(val_records),
        'sources':{k:choose_asset('val_'+k).label() for k in MODELS}}
    SELECT_FILE.write_text(json.dumps(FINAL_SELECTION,indent=2,ensure_ascii=False))
    if winner['name']=='Original WBF winner':
        print('NO IMPROVEMENT on full validation: keep original WBF submission.')
    else:
        print('TILED MODEL IMPROVED full validation mAP. Test inference is now worth considering.')
else:
    print('RUN_FULL=False: no expensive inference on 2,846 images.')
    if FULL_RESULTS_FILE.exists():display(pd.read_csv(FULL_RESULTS_FILE).sort_values('mAP',ascending=False))

val {'size': 640, 'overlap': 0.2}: 241 cached, 2605 missing of 2846 requested
FCOS loaded strictly: /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/model_weights.pth on cuda


Tiling val {'size': 640, 'overlap': 0.2}:   0%|          | 0/2605 [00:00<?, ?it/s]

wbf_r33_t33_f33_i65_max_s02:   0%|          | 0/2846 [00:00<?, ?it/s]

FULL factor 0.7 {'mAP': 0.18956931830686197, 'AP50': 0.3770047401000992, 'AP75': 0.1691649981767803, 'AP_small': 0.09752439341443228, 'AP_medium': 0.18310286225120198, 'AP_large': 0.32359192155802896}


wbf_r33_t33_f33_i65_max_s02:   0%|          | 0/2846 [00:00<?, ?it/s]

FULL factor 0.85 {'mAP': 0.1877011130453213, 'AP50': 0.37350552706954687, 'AP75': 0.16727049075454922, 'AP_small': 0.09635358063005724, 'AP_medium': 0.18261030306796386, 'AP_large': 0.32173575286779943}


wbf_r33_t33_f33_i65_max_s02:   0%|          | 0/2846 [00:00<?, ?it/s]

FULL factor 1.0 {'mAP': 0.18275625105920412, 'AP50': 0.3633620609637581, 'AP75': 0.16270589789078235, 'AP_small': 0.09179364226059886, 'AP_medium': 0.1797704485508614, 'AP_large': 0.3184834181386721}


,name,size,overlap,factor,mAP,AP50,AP75,AP_small,AP_medium,AP_large
1,WBF + tiled FCOS,640,0.2,0.70,0.189569,0.377005,0.169165,0.097524,0.183103,0.323592
2,WBF + tiled FCOS,640,0.2,0.85,0.187701,0.373506,0.167270,0.096354,0.182610,0.321736
0,Original WBF winner,0,0.0,0.00,0.186048,0.363248,0.168733,0.097091,0.177381,0.327260
3,WBF + tiled FCOS,640,0.2,1.00,0.182756,0.363362,0.162706,0.091794,0.179770,0.318483


TILED MODEL IMPROVED full validation mAP. Test inference is now worth considering.


### Результати Full Validation та вибір конфігурації

| Конфігурація | mAP | AP50 | AP_small | AP_medium | AP_large |
|---|---:|---:|---:|---:|---:|
| WBF original | 0.186048 | 0.363248 | 0.097091 | 0.177381 | **0.327260** |
| WBF + tiled FCOS, factor 0.70 | **0.189569** | **0.377005** | **0.097524** | **0.183103** | 0.323592 |
| WBF + tiled FCOS, factor 0.85 | 0.187701 | 0.373506 | 0.096354 | 0.182610 | 0.321736 |
| WBF + tiled FCOS, factor 1.00 | 0.182756 | 0.363362 | 0.091794 | 0.179770 | 0.318483 |

Найкраще показала себе конфігурація **640×640, overlap 20%, factor 0.70**. Вона підвищила повний validation mAP приблизно на **0.003521**. Приріст `AP_small` невеликий, помітніший він для `AP_medium`, а `AP_large` дещо погіршився. Це підтверджує користь підходу для загального mAP, але не доводить значного покращення саме малих об'єктів.

## 8. Test Inference 



In [16]:
RUN_TEST = True

In [13]:
def find_test_metadata():
    opts=list(DATA_ROOT.rglob('test_images.csv'))
    if not opts:opts=list(INPUT.rglob('test_images.csv'))
    if len(opts)!=1:raise FileNotFoundError(f'Missing/ambiguous test_images.csv: {opts}')
    return opts[0]

def make_submission(records,preds,destination):
    by_id=grouped(preds);rows=[]
    for rec in records:
        iid=str(rec['image_id']);W=float(rec['width']);H=float(rec['height'])
        words=[]
        for p in sorted(by_id.get(iid,[]),key=lambda z:z['score'],reverse=True)[:OUTPUT_MAX_DET]:
            x,y,w,h=map(float,p['bbox'])
            x=max(0.,min(W,x));y=max(0.,min(H,y))
            w=max(0.,min(w,W-x));h=max(0.,min(h,H-y))
            if w<=0 or h<=0:continue
            cls=int(p['category_id'])-1
            assert 0<=cls<=7
            words.append(f'{cls} {float(p["score"]):.6f} {x:.2f} {y:.2f} {w:.2f} {h:.2f}')
        rows.append({'image_id':iid,'PredictionString':' '.join(words)})
    df=pd.DataFrame(rows,columns=['image_id','PredictionString'])
    assert len(df)==len(records) and df.image_id.is_unique
    assert all(len(x.split())%6==0 for x in df.PredictionString)
    df.to_csv(destination,index=False)
    print('Submission rows:',len(df),'saved:',destination)
    return df

if RUN_TEST:
    assert SELECT_FILE.exists(), 'Run full validation before generating any new submission.'
    winner=json.loads(SELECT_FILE.read_text())['selected']
    source=find_test_metadata()
    testdf=pd.read_csv(source,dtype={'image_id':str},keep_default_na=False)
    assert {'image_id','file_name','width','height'}.issubset(testdf.columns)
    assert testdf.image_id.is_unique
    records=[{'image_id':str(r['image_id']),'file_name':str(r['file_name']),
        'width':int(r['width']),'height':int(r['height'])} for r in testdf.to_dict('records')]
    test_groups={}
    for m in MODELS:
        test_groups[m]=grouped(normalize_preds(read_asset_json(choose_asset('test_'+m,required=True)),'test'))
    if winner['name']=='WBF + tiled FCOS':
        cfg={'size':int(winner['size']),'overlap':float(winner['overlap'])}
        # Tiled test GPU; all partial results are cached during processing.
        cropped=tiles_for_records(records,'test',cfg)
        full_plus=merge_full_plus_tiles(records,test_groups['fcos'],cropped,scale=float(winner['factor']))
        test_groups['fcos']=grouped(full_plus)
    predictions=wbf_records(records,BEST_SPEC,test_groups,progress=True)
    write_gz(OUT/'test_selected_wbf.json.gz',predictions)
    df=make_submission(records,predictions,WORK/'submission.csv')
    (OUT/'submission_selection.json').write_text(json.dumps(winner,indent=2))
    print('Submit /kaggle/working/submission.csv only after checking its format.')
else:
    print('RUN_TEST=False: no test inference or submission is generated.')

test {'size': 640, 'overlap': 0.2}: 2400 cached, 1411 missing of 3811 requested
FCOS loaded strictly: /kaggle/input/datasets/olenadovbenchuk/result-of-ensemble/model_weights.pth on cuda


Tiling test {'size': 640, 'overlap': 0.2}:   0%|          | 0/1411 [00:00<?, ?it/s]

wbf_r33_t33_f33_i65_max_s02:   0%|          | 0/3811 [00:00<?, ?it/s]

Submission rows: 3811 saved: /kaggle/working/submission.csv
Submit /kaggle/working/submission.csv only after checking its format.


## Висновки

Експеримент перевіряв, чи може **FCOS Tiled Inference** покращити готовий ансамбль RetinaNet TTA + RT-DETR-L + FCOS. Для цього використовували фрагменти `640×640` із 20% overlap та змінювали лише вагу впливу tiled-прогнозів (`factor`).

**Результати:**

- **Pilot (241 кадр):** mAP ансамблю виріс із `0.197136` до `0.200599` для `factor=0.70`.
- **Full Validation (2846 кадрів):** найкращий `factor=0.70` дав **mAP `0.189569`** порівняно з **`0.186048`** для звичайного WBF (**+0.003521 mAP; ≈+1.89% відносно**).
- **Малі об'єкти:** на повному validation `AP_small` змінився з `0.097091` до `0.097524` — це невелике покращення. Натомість `AP_medium` підвищився, а `AP_large` трохи знизився.
- **Вибір factor має значення:** `factor=1.00` погіршив full-validation mAP до `0.182756`. Отже, більше tiled detections не означає автоматично кращу якість.


**Висновок:** tiled inference дав невелике, але покращення якості ансамблю на повній валідації. 